# ▶ Run everything

**First time:** *Run → Run All Cells*. Step 1 trains and attacks every model (notebooks 00 → 09, a few hours on a GPU), then Step 2 opens the app.

**Every time after that (e.g. demo day):** run **only the Step 2 cell**. It opens the app on the saved results in seconds, with no training.

Training is done once: models and results are saved in `data/`, `models/` and `results/`. Re-running Step 1 is safe because finished models and attack runs are skipped.

## Step 1 · Train everything (one time)
Runs notebooks 00 → 09 in order and saves each one's outputs, so you can open any of them afterwards to see its tables and figures. Progress is printed below; the long cells are YOLO training (03, 04).

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
env = {**os.environ, "PYTHONPATH": str(ROOT / "src"), "PYTHONIOENCODING": "utf-8"}
proc = subprocess.Popen([sys.executable, "-u", "-m", "mia.pipeline"], cwd=ROOT, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
for line in proc.stdout:
    if "IPKernelApp" not in line:  # harmless per-notebook kernel notice
        print(line, end="", flush=True)
print("\nTraining pipeline finished." if proc.wait() == 0 else "\nPipeline stopped with an error (see above).")

## Step 2 · Launch the app (run only this cell on demo day)
Starts Streamlit (http://localhost:8501, or the next free port) and opens it in your browser. Keep this notebook open while presenting.

In [ ]:
import socket, subprocess, sys, time, webbrowser
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def port_busy(port):
    with socket.socket() as s:
        return s.connect_ex(("localhost", port)) == 0


if not (ROOT / "results" / "mia.sqlite").exists():
    print("No results found yet. Run the Step 1 cell first (one time).")
elif "app" in globals() and app.poll() is None:
    print(f"The app is already running: {URL}")
    webbrowser.open(URL)
else:
    port = next(p for p in range(8501, 8600) if not port_busy(p))  # skip ports other apps are using
    URL = f"http://localhost:{port}"
    app = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(ROOT / "app" / "Home.py"),
                            "--server.port", str(port), "--server.headless", "true"], cwd=ROOT)
    for _ in range(60):  # wait up to 30 s for the server to come up
        time.sleep(0.5)
        if port_busy(port):
            break
    webbrowser.open(URL)
    print(f"App running: {URL}  (opened in your browser; run the Step 3 cell to stop it)")

## Step 3 · Stop the app (optional)

In [ ]:
if "app" in globals() and app.poll() is None:
    app.terminate()
    print("App stopped.")
else:
    print("No app started from this notebook is running.")